In [1]:
import numpy as np
import pandas as pd

import torch

import matplotlib.pyplot as plt

## Load Train Dataset

In [2]:
df = pd.read_csv('../data/train.csv')
df = df[['PassengerId', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Pclass', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'Survived']]

df

,PassengerId,Name,Sex,Age,SibSp,Parch,Pclass,Ticket,Fare,Cabin,Embarked,Survived
0,1,"Braund, Mr. Owen Harris",male,22.0,1,0,3,A/5 21171,7.2500,NaN,S,0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,1,PC 17599,71.2833,C85,C,1
2,3,"Heikkinen, Miss. Laina",female,26.0,0,0,3,STON/O2. 3101282,7.9250,NaN,S,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,1,113803,53.1000,C123,S,1
4,5,"Allen, Mr. William Henry",male,35.0,0,0,3,373450,8.0500,NaN,S,0
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,"Montvila, Rev. Juozas",male,27.0,0,0,2,211536,13.0000,NaN,S,0
887,888,"Graham, Miss. Margaret Edith",female,19.0,0,0,1,112053,30.0000,B42,S,1
888,889,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,3,W./C. 6607,23.4500,NaN,S,0
889,890,"Behr, Mr. Karl Howell",male,26.0,0,0,1,111369,30.0000,C148,C,1


## Train/Val Split

In [3]:
df_val = df.groupby('Survived').sample(frac=0.2, random_state=42)
df = df.drop(index=df_val.index)

len(df), len(df_val)

(713, 178)

## NULL Values

In [4]:
df.isna().sum()

PassengerId      0
Name             0
Sex              0
Age            132
SibSp            0
Parch            0
Pclass           0
Ticket           0
Fare             0
Cabin          552
Embarked         1
Survived         0
dtype: int64

### Age

In [5]:
df[['Age']].describe()

,Age
count,581.000000
mean,29.147745
std,14.359043
min,0.420000
25%,20.000000
50%,28.000000
75%,37.000000
max,74.000000


In [6]:
age_mean = df['Age'].mean()

df['Age'] = df['Age'].fillna(age_mean)
df_val['Age'] = df_val['Age'].fillna(age_mean)

df['Age'].isna().sum(), df_val['Age'].isna().sum()

(np.int64(0), np.int64(0))

### Cabin

In [7]:
df[['Cabin']].describe()

,Cabin
count,161
unique,116
top,C23 C25 C27
freq,4


In [8]:
df['Cabin'].value_counts()

Cabin
C23 C25 C27    4
B96 B98        4
G6             3
F33            3
F2             3
              ..
D28            1
E17            1
A24            1
C50            1
B42            1
Name: count, Length: 116, dtype: int64

In [9]:
for d in (df, df_val):
    d['Deck'] = d['Cabin'].str.split().str[-1].str[0].fillna('unknown')
    d['CabinCount'] = d['Cabin'].str.split().str.len().fillna(0).astype(int)

In [10]:
df[['Deck']].describe()

,Deck
count,713
unique,9
top,unknown
freq,552


In [11]:
df['CabinCount'].value_counts()

CabinCount
0    552
1    139
2     14
3      6
4      2
Name: count, dtype: int64

In [12]:
cols = ['PassengerId', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Pclass', 'Ticket', 'Fare', 'Deck', 'CabinCount', 'Embarked', 'Survived']
df = df[cols]
df_val = df_val[cols]
df

,PassengerId,Name,Sex,Age,SibSp,Parch,Pclass,Ticket,Fare,Deck,CabinCount,Embarked,Survived
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000000,1,0,1,PC 17599,71.2833,C,1,C,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000000,1,0,1,113803,53.1000,C,1,S,1
4,5,"Allen, Mr. William Henry",male,35.000000,0,0,3,373450,8.0500,unknown,0,S,0
6,7,"McCarthy, Mr. Timothy J",male,54.000000,0,0,1,17463,51.8625,E,1,S,0
7,8,"Palsson, Master. Gosta Leonard",male,2.000000,3,1,3,349909,21.0750,unknown,0,S,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,884,"Banfield, Mr. Frederick James",male,28.000000,0,0,2,C.A./SOTON 34068,10.5000,unknown,0,S,0
886,887,"Montvila, Rev. Juozas",male,27.000000,0,0,2,211536,13.0000,unknown,0,S,0
887,888,"Graham, Miss. Margaret Edith",female,19.000000,0,0,1,112053,30.0000,B,1,S,1
888,889,"Johnston, Miss. Catherine Helen ""Carrie""",female,29.147745,1,2,3,W./C. 6607,23.4500,unknown,0,S,0


### Embarked

In [13]:
df[['Embarked']].describe()

,Embarked
count,712
unique,3
top,S
freq,527


In [14]:
df['Embarked'].value_counts()

Embarked
S    527
C    128
Q     57
Name: count, dtype: int64

In [15]:
embarked_mode = df['Embarked'].mode()[0]

df['Embarked'] = df['Embarked'].fillna(embarked_mode)
df_val['Embarked'] = df_val['Embarked'].fillna(embarked_mode)

In [16]:
pd.DataFrame({'train': df.isna().sum(), 'val': df_val.isna().sum()})

,train,val
PassengerId,0,0
Name,0,0
Sex,0,0
Age,0,0
SibSp,0,0
Parch,0,0
Pclass,0,0
Ticket,0,0
Fare,0,0
Deck,0,0


## Standardization

In [17]:
df_Y = df['Survived']
df_X = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Survived'])

df_val_Y = df_val['Survived']
df_val_X = df_val.drop(columns=['PassengerId', 'Name', 'Ticket', 'Survived'])

In [18]:
df_X.dtypes

Sex               str
Age           float64
SibSp           int64
Parch           int64
Pclass          int64
Fare          float64
Deck           object
CabinCount      int64
Embarked          str
dtype: object

In [19]:
ordinal_cols = df_X.select_dtypes(include='number').columns.tolist()

ordinal_cols

['Age', 'SibSp', 'Parch', 'Pclass', 'Fare', 'CabinCount']

In [20]:
categorical_cols = df_X.select_dtypes(include=['object', 'string']).columns.tolist()

categorical_cols

['Sex', 'Deck', 'Embarked']

### Ordinal Columns

In [21]:
df_X[ordinal_cols].describe()

,Age,SibSp,Parch,Pclass,Fare,CabinCount
count,713.000000,713.000000,713.000000,713.000000,713.000000,713.000000
mean,29.147745,0.532959,0.375877,2.305750,32.756755,0.270687
std,12.959836,1.096134,0.769197,0.830719,50.047042,0.566823
min,0.420000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,22.000000,0.000000,0.000000,2.000000,7.925000,0.000000
50%,29.147745,0.000000,0.000000,3.000000,14.454200,0.000000
75%,35.000000,1.000000,0.000000,3.000000,31.387500,0.000000
max,74.000000,8.000000,6.000000,3.000000,512.329200,4.000000


In [22]:
# Calculate statistics from training data only
means = df_X[ordinal_cols].mean()
stds = df_X[ordinal_cols].std()

# Standardize both datasets
df_X[ordinal_cols] = (df_X[ordinal_cols] - means) / stds
df_val_X[ordinal_cols] = (df_val_X[ordinal_cols] - means) / stds

In [23]:
df_X[ordinal_cols].describe()

,Age,SibSp,Parch,Pclass,Fare,CabinCount
count,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02
mean,-1.893452e-16,9.965536e-18,7.162729e-17,1.918366e-16,2.491384e-18,7.474152e-18
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-2.216675e+00,-4.862172e-01,-4.886608e-01,-1.571832e+00,-6.545193e-01,-4.775515e-01
25%,-5.515305e-01,-4.862172e-01,-4.886608e-01,-3.680553e-01,-4.961683e-01,-4.775515e-01
50%,-2.741326e-16,-4.862172e-01,-4.886608e-01,8.357219e-01,-3.657070e-01,-4.775515e-01
75%,4.515686e-01,4.260798e-01,-4.886608e-01,8.357219e-01,-2.735935e-02,-4.775515e-01
max,3.460866e+00,6.812158e+00,7.311679e+00,8.357219e-01,9.582433e+00,6.579324e+00


### Categorical Columns

In [24]:
df_X[categorical_cols].describe()

,Sex,Deck,Embarked
count,713,713,713
unique,2,9,3
top,male,unknown,S
freq,464,552,528


In [25]:
df_X = pd.get_dummies(df_X, columns=categorical_cols, dtype=int)
df_val_X = pd.get_dummies(df_val_X, columns=categorical_cols, dtype=int)
df_val_X = df_val_X.reindex(columns=df_X.columns, fill_value=0)

df_X

,Age,SibSp,Parch,Pclass,Fare,CabinCount,Sex_female,Sex_male,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_unknown,Embarked_C,Embarked_Q,Embarked_S
1,6.830530e-01,0.426080,-0.488661,-1.571832,0.769807,1.286667,1,0,0,0,1,0,0,0,0,0,0,1,0,0
3,4.515686e-01,0.426080,-0.488661,-1.571832,0.406482,1.286667,1,0,0,0,1,0,0,0,0,0,0,0,0,1
4,4.515686e-01,-0.486217,-0.488661,0.835722,-0.493671,-0.477552,0,1,0,0,0,0,0,0,0,0,1,0,0,1
6,1.917637e+00,-0.486217,-0.488661,-1.571832,0.381756,1.286667,0,1,0,0,0,0,1,0,0,0,0,0,0,1
7,-2.094760e+00,2.250674,0.811396,0.835722,-0.233415,-0.477552,0,1,0,0,0,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,-8.856171e-02,-0.486217,-0.488661,-0.368055,-0.444717,-0.477552,0,1,0,0,0,0,0,0,0,0,1,0,0,1
886,-1.657232e-01,-0.486217,-0.488661,-0.368055,-0.394764,-0.477552,0,1,0,0,0,0,0,0,0,0,1,0,0,1
887,-7.830149e-01,-0.486217,-0.488661,-1.571832,-0.055083,1.286667,1,0,0,1,0,0,0,0,0,0,0,0,0,1
888,-2.741326e-16,0.426080,2.111452,0.835722,-0.185960,-0.477552,1,0,0,0,0,0,0,0,0,0,1,0,0,1
